In [2]:
import sqlite3
import pandas as pd
import numpy as np

from pathlib import Path

# Đường dẫn database
DB_PATH = Path("../data/database/career.db")

# Nếu notebook nằm trong thư mục con, hãy chỉnh lại DB_PATH
conn = sqlite3.connect(DB_PATH)

# 1. Đọc Work Activities
activities = pd.read_sql_query(
    """
    SELECT
        occupation_code,
        activity,
        level_score
    FROM occupation_work_activities
    """,
    conn
)

# 2. Đọc AI Exposure
exposure = pd.read_sql_query(
    """
    SELECT
        occupation_code,
        occupation_title,
        ai_exposure_score
    FROM occupations
    """,
    conn
)

conn.close()

# 3. Kiểm tra dữ liệu ban đầu
print("Work Activities")
print("Rows:", len(activities))
print("Occupations:", activities["occupation_code"].nunique())
print("Activities:", activities["activity"].nunique())

print("\nAI Exposure")
print("Occupation records:", len(exposure))
print("Valid exposure:", exposure["ai_exposure_score"].notna().sum())
print("Missing exposure:", exposure["ai_exposure_score"].isna().sum())

print("\nDuplicate occupation-activity pairs:",
      activities.duplicated(["occupation_code", "activity"]).sum())

Work Activities
Rows: 37351
Occupations: 911
Activities: 41

AI Exposure
Occupation records: 1016
Valid exposure: 879
Missing exposure: 137

Duplicate occupation-activity pairs: 0


In [3]:
# Mỗi hàng là một nghề, mỗi cột là một Work Activity
X = activities.pivot(
    index="occupation_code",
    columns="activity",
    values="level_score"
)

# Đặt tên index để dễ theo dõi
X.index.name = "occupation_code"

# Ghép AI Exposure theo occupation_code
df = X.join(
    exposure.set_index("occupation_code")["ai_exposure_score"],
    how="inner"
)

# Kiểm tra dữ liệu
activity_cols = X.columns.tolist()

print("Matrix shape:", X.shape)
print("Number of activities:", len(activity_cols))
print("Joined dataset:", df.shape)

print("\nMissing values in activity features:",
      df[activity_cols].isna().sum().sum())

print("Missing AI Exposure:",
      df["ai_exposure_score"].isna().sum())

print("\nPreview:")
display(df.head())

Matrix shape: (911, 41)
Number of activities: 41
Joined dataset: (911, 42)

Missing values in activity features: 0
Missing AI Exposure: 32

Preview:


,Analyzing Data or Information,Assisting and Caring for Others,Coaching and Developing Others,Communicating with People Outside the Organization,"Communicating with Supervisors, Peers, or Subordinates",Controlling Machines and Processes,Coordinating the Work and Activities of Others,Developing Objectives and Strategies,Developing and Building Teams,Documenting/Recording Information,...,Repairing and Maintaining Mechanical Equipment,Resolving Conflicts and Negotiating with Others,Scheduling Work and Activities,Selling or Influencing Others,Staffing Organizational Units,Thinking Creatively,Training and Teaching Others,Updating and Using Relevant Knowledge,Working with Computers,ai_exposure_score
occupation_code,,,,,,,,,,,,,,,,,,,,,
11-1011.00,4.84,3.10,5.85,5.47,5.82,1.82,5.60,5.46,5.38,3.80,...,0.27,5.74,4.41,4.91,4.80,4.45,4.76,5.09,3.03,0.491063
11-1011.03,4.81,2.67,4.78,6.11,5.93,1.00,5.04,5.56,4.81,3.41,...,0.78,4.70,4.93,5.07,4.26,4.85,4.22,5.59,3.22,0.696279
11-1021.00,4.27,3.03,4.53,4.42,4.85,1.88,4.62,3.54,3.88,3.37,...,1.91,4.89,3.92,3.76,4.33,3.95,3.79,4.17,4.15,0.739312
11-2011.00,2.85,2.95,4.33,4.76,5.28,0.51,4.24,3.94,3.76,3.75,...,0.23,4.38,4.11,3.58,3.03,5.15,2.80,4.36,3.59,0.738163
11-2021.00,4.51,3.18,5.03,5.81,5.31,1.88,4.55,5.26,4.02,3.71,...,1.20,5.17,5.05,6.12,4.17,5.77,3.72,5.67,4.13,0.742922


In [4]:
from scipy.stats import spearmanr

# Chỉ giữ các nghề có đủ dữ liệu cho cả 41 LV và AI Exposure
analysis_df = df.dropna(
    subset=activity_cols + ["ai_exposure_score"]
).copy()

print("Số nghề được phân tích:", len(analysis_df))

# Tính tương quan Spearman cho từng hoạt động
results = []

for activity in activity_cols:
    rho, p_value = spearmanr(
        analysis_df[activity],
        analysis_df["ai_exposure_score"]
    )

    results.append({
        "activity": activity,
        "spearman_rho": rho,
        "p_value": p_value
    })

corr_df = pd.DataFrame(results)

# Sắp xếp theo độ lớn tuyệt đối của tương quan
corr_df["abs_rho"] = corr_df["spearman_rho"].abs()

corr_df = corr_df.sort_values(
    "abs_rho",
    ascending=False
).reset_index(drop=True)

display(corr_df)

Số nghề được phân tích: 879


,activity,spearman_rho,p_value,abs_rho
0,Developing Objectives and Strategies,-0.650225,9.135923e-107,0.650225
1,Making Decisions and Solving Problems,-0.628524,8.270284e-98,0.628524
2,Establishing and Maintaining Interpersonal Rel...,-0.625388,1.424013e-96,0.625388
3,Coaching and Developing Others,-0.624958,2.099404e-96,0.624958
4,Training and Teaching Others,-0.617542,1.531273e-93,0.617542
5,Providing Consultation and Advice to Others,-0.604827,8.303548e-89,0.604827
6,"Organizing, Planning, and Prioritizing Work",-0.593235,1.129773e-84,0.593235
7,Communicating with People Outside the Organiza...,-0.591258,5.511970e-84,0.591258
8,Thinking Creatively,-0.579310,6.334178e-80,0.579310
9,Getting Information,-0.576935,3.882712e-79,0.576935
